## Previsão de Rotatividade de Clientes
### Projeto Didático — Data Science & Machine Learning Clássico

---

**Objetivo:** construir um pipeline completo de Data Science para prever quais clientes de uma empresa de telecomunicações têm maior probabilidade de cancelar o serviço (*churn*), permitindo ações de retenção proativas.

**Dataset:** [Telco Customer Churn](https://www.kaggle.com/blastchar/telco-customer-churn) (IBM Sample Dataset) — 7.043 clientes, 21 atributos.

**O que este notebook cobre:**
1. Contextualização do problema de negócio
2. Análise Exploratória de Dados (EDA)
3. Limpeza e pré-processamento
4. Engenharia de features
5. Tratamento de desbalanceamento de classes
6. Modelagem: Regressão Logística → Random Forest → XGBoost
7. Avaliação com métricas apropriadas (não só acurácia!)
8. Interpretação de resultados e importância das variáveis
9. Conclusões e próximos passos

## 1. Contexto do Problema de Negócio

**Churn** (ou taxa de cancelamento) é uma das métricas mais críticas para empresas baseadas em assinatura (telecom, streaming, SaaS, etc.). Adquirir um novo cliente custa, em média, de 5 a 25 vezes mais do que reter um cliente existente.

**Pergunta de negócio:** *Dado o perfil e o histórico de um cliente, qual a probabilidade de ele cancelar o serviço no próximo período?*

**Por que isso importa:**
- Permite à empresa priorizar ações de retenção (descontos, contato proativo, upgrade de plano) para clientes de alto risco.
- Evita desperdiçar recursos de retenção em clientes que não iriam cancelar de qualquer forma.

**Tipo de problema:** Classificação binária supervisionada (`Churn` = Yes/No).

**Desafio central deste dataset:** as classes são desbalanceadas (~73% não-churn vs ~27% churn), então acurácia sozinha é uma métrica enganosa — um modelo "preguiçoso" que sempre prevê "não vai cancelar" já acerta 73% das vezes sem aprender nada útil.


## 2. Setup do Ambiente

Importar as bibliotecas necessárias. Se estiver rodando localmente pela primeira vez, instale as dependências:

```bash
pip install pandas numpy matplotlib seaborn scikit-learn xgboost imbalanced-learn shap jinja2
```


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# Configurações visuais
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

# Reprodutibilidade
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print('Ambiente configurado com sucesso.')

## 3. Carregamento dos Dados

O dataset "Telco Customer Churn" é público (IBM Sample Data Sets). Neste projeto vamos carregá-lo diretamente.


In [ ]:
# Caminho do arquivo local (ajuste conforme necessário)
CAMINHO_CSV = 'Telco-Customer-Churn.csv'

df = pd.read_csv(CAMINHO_CSV)

print(f'Shape do dataset: {df.shape[0]} linhas x {df.shape[1]} colunas')
df.head()

In [ ]:
df.info()

In [ ]:
# Estatísticas descritivas das variáveis numéricas
df.describe()

## 4. Limpeza dos Dados

Alguns pontos de atenção conhecidos neste dataset (bugs "clássicos" que todo cientista de dados deve saber identificar):

1. **`TotalCharges`** vem como `object` (string), não `float` — porque existem 11 registros com espaço em branco `" "` no lugar de um número (clientes com `tenure = 0`, ou seja, cadastrados mas ainda sem cobrança).
2. **`customerID`** é um identificador único, sem valor preditivo — deve ser removido antes da modelagem (mas guardado para referência).
3. Várias colunas categóricas usam `"No internet service"` / `"No phone service"` como uma categoria distinta de `"No"` — vamos decidir conscientemente se simplificamos isso.


In [ ]:
# 1. Corrigindo TotalCharges
print('Valores não numéricos em TotalCharges:', (df['TotalCharges'].str.strip() == '').sum())

# Converte para numérico, forçando erros para NaN
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

print(f'Valores nulos após conversão: {df["TotalCharges"].isnull().sum()}')
print(df[df['TotalCharges'].isnull()][['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges']])

In [ ]:
# Confirma a hipótese: todos os NaN são clientes com tenure == 0 (acabaram de entrar)
assert (df.loc[df['TotalCharges'].isnull(), 'tenure'] == 0).all(), 'Hipótese não confirmada!'

# Para esses clientes, faz sentido que TotalCharges seja 0 (ainda não foram cobrados)
df['TotalCharges'] = df['TotalCharges'].fillna(0)

print('Nulos restantes em TotalCharges:', df['TotalCharges'].isnull().sum())

In [ ]:
# 2. Guardamos customerID à parte e removemos do dataframe de trabalho
customer_ids = df['customerID'].copy()
df = df.drop(columns=['customerID'])

# 3. Verificando duplicatas
print('Linhas duplicadas:', df.duplicated().sum())

# 4. Checagem final de nulos
df.isnull().sum()[df.isnull().sum() > 0]

## 5. Análise Exploratória de Dados (EDA)

### 5.1 Distribuição da variável-alvo (Churn)


In [ ]:
churn_counts = df['Churn'].value_counts()
churn_pct = df['Churn'].value_counts(normalize=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(churn_counts.index, churn_counts.values, color=['#2ecc71', '#e74c3c'])
axes[0].set_title('Distribuição Absoluta de Churn')
axes[0].set_ylabel('Número de Clientes')
for i, v in enumerate(churn_counts.values):
    axes[0].text(i, v + 50, str(v), ha='center', fontweight='bold')

axes[1].pie(churn_pct.values, labels=churn_pct.index, autopct='%1.1f%%',
            colors=['#2ecc71', '#e74c3c'], startangle=90)
axes[1].set_title('Distribuição Percentual de Churn')

plt.tight_layout()
plt.show()

print(f'\nDesbalanceamento de classes: {churn_pct["No"]:.1f}% não-churn vs {churn_pct["Yes"]:.1f}% churn')
print('=> Isso confirma que acurácia sozinha NÃO é uma métrica confiável para este problema.')

### 5.2 Churn por variáveis categóricas relevantes

Vamos investigar como o churn se relaciona com tipo de contrato, método de pagamento e serviço de internet — hipóteses de negócio comuns.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 11))

categorical_vars = ['Contract', 'PaymentMethod', 'InternetService', 'PaperlessBilling']

for ax, col in zip(axes.flatten(), categorical_vars):
    churn_rate = df.groupby(col)['Churn'].apply(lambda x: (x == 'Yes').mean() * 100).sort_values(ascending=False)
    bars = ax.bar(churn_rate.index, churn_rate.values, color='#3498db')
    ax.set_title(f'Taxa de Churn por {col}')
    ax.set_ylabel('Taxa de Churn (%)')
    ax.tick_params(axis='x', rotation=30)
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.5, f'{h:.1f}%', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

**Insights esperados (devemos validar ao rodar):**
- Contratos `Month-to-month` tendem a ter churn muito mais alto que contratos anuais/bianuais (faz sentido: menor barreira de saída).
- `Electronic check` como método de pagamento costuma ter maior churn — pode indicar um segmento de clientes menos fidelizado.
- Fiber optic tende a ter mais churn que DSL — possivelmente por preço mais alto ou problemas de qualidade percebidos.


In [ ]:
# Distribuição de variáveis numéricas por churn
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

numeric_vars = ['tenure', 'MonthlyCharges', 'TotalCharges']

for ax, col in zip(axes, numeric_vars):
    sns.kdeplot(data=df, x=col, hue='Churn', fill=True, alpha=0.4, ax=ax,
                palette={'No': '#2ecc71', 'Yes': '#e74c3c'})
    ax.set_title(f'Distribuição de {col} por Churn')

plt.tight_layout()
plt.show()

**Insight chave:** clientes com **baixo `tenure`** (poucos meses de casa) têm risco de churn muito maior. Isso sugere que o período inicial do cliente é crítico para retenção — um achado clássico e acionável neste tipo de negócio.


In [ ]:
# Matriz de correlação das variáveis numéricas
df_corr = df.copy()
df_corr['Churn_binary'] = (df_corr['Churn'] == 'Yes').astype(int)

corr_cols = ['tenure', 'MonthlyCharges', 'TotalCharges', 'SeniorCitizen', 'Churn_binary']
corr_matrix = df_corr[corr_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, fmt='.2f', linewidths=0.5)
plt.title('Matriz de Correlação (variáveis numéricas + Churn)')
plt.tight_layout()
plt.show()